# Weapon Detection — Test Error Analysis Notebook

**Purpose:** analyze the already-evaluated Epoch-46 candidate without retraining.

Current locked benchmark:

- Validation mAP50-95: **0.493**
- Test mAP50-95: **0.411**
- Test precision: **0.721**
- Test recall: **0.547**
- Test size: **784 images / 775 objects**

Important rule: the Test set has already been used for the final benchmark. This notebook is for **diagnosis**, not for repeated model selection.

## 1. One-time runtime preparation

Use the same pinned environment as the training notebook. Restore the archived dataset only when the local dataset is absent.

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

# Mount Drive
mountpoint = Path("/content/drive")
if not (mountpoint / "MyDrive").exists():
    if mountpoint.exists():
        import shutil
        shutil.rmtree(mountpoint)
    drive.mount("/content/drive")

# Ultralytics
try:
    import ultralytics
except Exception:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.123"]
    )
    import ultralytics

assert ultralytics.__version__ == "8.4.123"
print("Ultralytics:", ultralytics.__version__)

In [ ]:
from pathlib import Path
import subprocess

DRIVE = Path("/content/drive/MyDrive")
ZIP_PATH = DRIVE / "Final_Dataset_Split" / "Final_Dataset_Split.zip"
LOCAL_DATASET = Path("/content/Final_Dataset_Split")

if not (LOCAL_DATASET / "test" / "images").exists():
    subprocess.run(
        ["unzip", "-q", str(ZIP_PATH), "-d", "/content"],
        check=True
    )

DATA_YAML = LOCAL_DATASET / "dataset_colab.yaml"
DATA_YAML.write_text(
"""path: /content/Final_Dataset_Split
train: train/images
val: val/images
test: test/images
names:
  0: Knife
  1: Machette
  2: Handgun
  3: Longgun
  4: Other
nc: 5
"""
)
print("✅ Error-analysis dataset ready.")

## 2. Load the final model

The final candidate is the permanent `FINAL_CONTINUATION/checkpoints/best.pt`.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

DRIVE = Path("/content/drive/MyDrive")

MODEL_PATH = (
    DRIVE /
    "Weapon_Detection_Training" /
    "FINAL_CONTINUATION" /
    "checkpoints" /
    "best.pt"
)

model = YOLO(str(MODEL_PATH))
print("✅ Final candidate loaded:", MODEL_PATH)

## 3. Diagnostic A — official test metrics (reference only)

This cell reproduces the already recorded final benchmark if needed. It is **not** intended for repeated tuning.

In [ ]:
results = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=24,
    device=0,
    workers=4,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True,
    project="/content/wd_training",
    name="epoch46_test_reference",
    exist_ok=True,
)

print(f"Precision : {results.box.mp:.4f}")
print(f"Recall    : {results.box.mr:.4f}")
print(f"mAP50     : {results.box.map50:.4f}")
print(f"mAP50-95  : {results.box.map:.4f}")

## 4. Diagnostic B — prediction audit

Next, inspect:
- false negatives;
- false positives;
- wrong-class detections;
- low-confidence true positives;
- missed Handguns;
- missed Machettes;
- small/occluded/dark/blurred weapons.

The goal is to explain **why** the Test score is below Validation before deciding on any new training.

In [ ]:
# Placeholder for the detailed matching/error-analysis routine.
# Keep this cell isolated so the official Test benchmark is not mixed
# with model-selection logic.

print("Next: run the detailed matching analysis.")
print("Do not train from this notebook until the error analysis is reviewed.")

## 5. Decision framework after error analysis

Use Validation, not repeated Test optimization, for any new experiment.

Possible outcomes:

- **Mostly missed objects / low recall:** investigate augmentation, image scale, confidence behavior, or class-specific data coverage.
- **Systematic class confusion:** inspect class semantics and annotation consistency before retraining.
- **Poor localization:** inspect box quality and consider controlled resolution experiments.
- **Strong precision but low recall:** investigate whether the detector is too conservative for the deployment scenario.
- **Errors concentrated in difficult surveillance scenes:** consider whether more representative training data is a better intervention than more epochs.

Any new training experiment must receive a new run name and a permanent per-epoch checkpoint archive.